# PART 2 — 数値はどうやって予測する？
## 実習2：カリフォルニアの住宅価格を予測しよう

**Features → Regression → Numerical Prediction**

この実習では、scikit-learn の **California Housing Dataset** を使って、
地域の特徴から **住宅価格の中央値** を予測します。

今回は、まず **線形回帰（Linear Regression）** を使い、
そのあと **決定木回帰（Decision Tree Regressor）** と結果を比べます。

### この実習で学ぶこと
1. 入力 **X** と正解 **y** を確認する
2. データを学習用とテスト用に分ける
3. 線形回帰モデルで学習・予測する
4. **MAE / MSE / RMSE / R²** で評価する
5. 実際の値と予測値をグラフで確認する
6. 決定木回帰と比較する
7. 学習データとテストデータの結果から **過学習** を考える

> このデータは 1990年のカリフォルニア国勢調査をもとにした学習用データです。  
> 現在の住宅価格を表すデータではありません。


## 1. ライブラリを読み込む

今回使うライブラリを読み込みます。

- `pandas`：表形式のデータを扱う
- `numpy`：数値計算
- `matplotlib`：グラフを描く
- `scikit-learn`：データセット、回帰モデル、評価指標を使う


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## 2. California Housing Dataset を読み込む

scikit-learn の `fetch_california_housing()` は、実行環境によっては
データ配布サイトへのアクセスが拒否されて **HTTP 403 エラー** になることがあります。

この実習では、同じ California Housing Dataset の CSV ミラーを
`pandas.read_csv()` で直接読み込みます。

Google Colab では、下のセルをそのまま実行してください。


In [ ]:
# California Housing Dataset の CSV を読み込む
url = "https://raw.githubusercontent.com/javascriptdata/scikit.js/main/docs/static/data/california_housing.csv"

df = pd.read_csv(url)

print("行数と列数:", df.shape)
display(df.head())


## 3. データの内容を確認する

このデータには **20,640件** の地域データがあり、入力には **8個の特徴量** を使います。

| 特徴量 | 意味 |
|---|---|
| MedInc | 地域の所得の中央値 |
| HouseAge | 住宅築年数の中央値 |
| AveRooms | 1世帯あたりの平均部屋数 |
| AveBedrms | 1世帯あたりの平均寝室数 |
| Population | 地域の人口 |
| AveOccup | 1世帯あたりの平均居住人数 |
| Latitude | 緯度 |
| Longitude | 経度 |

正解 `MedHouseVal` は、その地域の **住宅価格の中央値** です。

### 価格の単位に注意
`MedHouseVal = 2.5` の場合、

**2.5 × 100,000ドル = 250,000ドル**

を表します。

### 今回の X と y
- **X**：8個の特徴量
- **y**：住宅価格の中央値 `MedHouseVal`


In [ ]:
print("列名:")
print(df.columns.tolist())

print("\nデータ数:", len(df))
print("欠損値の数:", df.isnull().sum().sum())


## 4. 住宅価格の値を確認する

`MedHouseVal` は 10万ドル単位なので、
学生が分かりやすいようにドルに換算した列も作ってみます。


In [ ]:
price_check = pd.DataFrame({
    "MedHouseVal": df["MedHouseVal"].head(),
    "住宅価格（ドル）": (df["MedHouseVal"].head() * 100000).round(0)
})

display(price_check)


## 5. 入力 X と正解 y を分ける

`MedHouseVal` 以外の8列を入力 **X** にします。

`MedHouseVal` を正解 **y** にします。


In [ ]:
# 入力 X：8個の特徴量
X = df.drop(columns=["MedHouseVal"])

# 正解 y：住宅価格の中央値
y = df["MedHouseVal"]

print("X の形:", X.shape)
print("y の形:", y.shape)

display(X.head())
display(y.head())


## 6. 学習用データとテスト用データに分ける

データの **80%を学習用（Train）**、
**20%をテスト用（Test）** に分けます。

`random_state=42` を指定して、毎回同じ分け方になるようにします。


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("学習用データ:", len(X_train))
print("テスト用データ:", len(X_test))


# PART A — 線形回帰

まず、糖尿病の実習でも使った **LinearRegression** で住宅価格を予測します。


## 7. 線形回帰モデルを学習する

`LinearRegression()` でモデルを作り、
`fit()` に学習用の X と y を渡します。


In [ ]:
# 線形回帰モデルを作る
linear_model = LinearRegression()

# 学習
linear_model.fit(X_train, y_train)

print("線形回帰の学習が完了しました。")


## 8. テストデータを予測する

学習に使っていない `X_test` をモデルに入力します。

`predict()` の結果が住宅価格の **予測値** です。


In [ ]:
# テストデータを予測
linear_pred = linear_model.predict(X_test)

print("予測したデータ数:", len(linear_pred))
print("最初の5個の予測値:")
print(np.round(linear_pred[:5], 3))


## 9. 実際の価格と予測価格を比べる

`MedHouseVal` は10万ドル単位なので、
ここではドルに換算して比較してみます。


In [ ]:
linear_result = pd.DataFrame({
    "実際の値": y_test.values,
    "予測値": linear_pred
})

linear_result["実際の価格（ドル）"] = linear_result["実際の値"] * 100000
linear_result["予測価格（ドル）"] = linear_result["予測値"] * 100000

display(
    linear_result[
        ["実際の価格（ドル）", "予測価格（ドル）"]
    ].head(10).round(0)
)


## 10. 線形回帰モデルを評価する

今回も4つの評価指標を使います。

- **MAE（Mean Absolute Error）**：平均絶対誤差
- **MSE（Mean Squared Error）**：平均二乗誤差
- **RMSE（Root Mean Squared Error）**：二乗平均平方根誤差
- **R²（Coefficient of Determination）**：決定係数

MAE と RMSE は `MedHouseVal` と同じ単位です。  
このデータでは **1 = 100,000ドル** なので、ドルに換算した値も表示します。


In [ ]:
# 評価指標を計算
linear_mae = mean_absolute_error(y_test, linear_pred)
linear_mse = mean_squared_error(y_test, linear_pred)
linear_rmse = np.sqrt(linear_mse)
linear_r2 = r2_score(y_test, linear_pred)

print("Linear Regression")
print("MAE :", round(linear_mae, 3),
      "  → 約", round(linear_mae * 100000), "ドル")
print("MSE :", round(linear_mse, 3))
print("RMSE:", round(linear_rmse, 3),
      "  → 約", round(linear_rmse * 100000), "ドル")
print("R²  :", round(linear_r2, 3))


## 11. 実際の値と予測値をグラフで確認する

横軸が **実際の値**、縦軸が **予測値** です。

点が赤い対角線に近いほど、実際の値に近い予測ができています。


In [ ]:
plt.figure(figsize=(7, 6))

plt.scatter(y_test, linear_pred, alpha=0.4, s=15)

min_value = min(y_test.min(), linear_pred.min())
max_value = max(y_test.max(), linear_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    "r--",
    label="Actual = Predicted"
)

plt.xlabel("Actual value")
plt.ylabel("Predicted value")
plt.title("Linear Regression: Actual vs Predicted")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


# PART B — 決定木回帰

前回の分類では **決定木（Decision Tree）** を使いました。

決定木は分類だけでなく、
**数値を予測する回帰**にも使うことができます。

ここでは `DecisionTreeRegressor` を使って、
線形回帰と結果を比べてみます。


## 12. 決定木回帰モデルを学習する

まず、木の深さを `max_depth=5` に制限した決定木を作ります。


In [ ]:
# 決定木回帰モデルを作る
tree_model = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

# 学習
tree_model.fit(X_train, y_train)

# テストデータを予測
tree_pred = tree_model.predict(X_test)

print("決定木回帰の学習と予測が完了しました。")


## 13. 決定木回帰を評価する

線形回帰と同じ指標を使って評価します。


In [ ]:
tree_mae = mean_absolute_error(y_test, tree_pred)
tree_mse = mean_squared_error(y_test, tree_pred)
tree_rmse = np.sqrt(tree_mse)
tree_r2 = r2_score(y_test, tree_pred)

print("Decision Tree Regressor")
print("MAE :", round(tree_mae, 3),
      "  → 約", round(tree_mae * 100000), "ドル")
print("MSE :", round(tree_mse, 3))
print("RMSE:", round(tree_rmse, 3),
      "  → 約", round(tree_rmse * 100000), "ドル")
print("R²  :", round(tree_r2, 3))


## 14. 2つのモデルを比べる

同じテストデータに対する結果を並べてみましょう。

- MAE / RMSE：小さいほどよい
- R²：一般的には 1 に近いほどよい


In [ ]:
comparison = pd.DataFrame({
    "モデル": ["Linear Regression", "Decision Tree"],
    "MAE": [linear_mae, tree_mae],
    "RMSE": [linear_rmse, tree_rmse],
    "R²": [linear_r2, tree_r2]
})

display(comparison.round(3))


## 15. 過学習を確認する

決定木は木を深くすると、
学習データに非常によく合わせることができます。

しかし、学習データに合わせすぎると、
テストデータの予測が悪くなることがあります。

これが **過学習（Overfitting）** です。

ここでは、深さを制限しない決定木を作り、
学習用とテスト用の R² を比較します。


In [ ]:
# 深さを制限しない決定木
deep_tree = DecisionTreeRegressor(random_state=42)

deep_tree.fit(X_train, y_train)

# 学習データとテストデータの R²
train_r2 = deep_tree.score(X_train, y_train)
test_r2 = deep_tree.score(X_test, y_test)

print("深さを制限しない決定木")
print("Train R²:", round(train_r2, 3))
print("Test  R²:", round(test_r2, 3))


### 結果の見方

もし、

- Train R² は非常に高い
- Test R² はそれよりかなり低い

という結果になった場合、

**学習データに合わせすぎている可能性がある**

と考えることができます。

学習データの成績だけでモデルの性能を判断してはいけません。


## 16. 学習に使わなかった1件を予測する

テストデータから1件だけ取り出して、
住宅価格を予測してみましょう。


In [ ]:
# テストデータから1行だけ取り出す
one_house = X_test.iloc[[0]]

# 線形回帰で予測
one_prediction = linear_model.predict(one_house)[0]

print("予測価格:",
      round(one_prediction * 100000), "ドル")

print("実際の価格:",
      round(y_test.iloc[0] * 100000), "ドル")


# まとめ

今回の実習では、カリフォルニア住宅データを使って
**住宅価格という連続した数値**を予測しました。

### 回帰の基本手順

**1. データを確認する**  
入力 X と正解 y は何か？

↓

**2. データを分割する**  
学習用とテスト用に分ける

↓

**3. モデルを学習する**  
`fit()`

↓

**4. 予測する**  
`predict()`

↓

**5. 評価する**  
MAE / MSE / RMSE / R²

### 今回さらに学んだこと

- 同じ回帰問題でも、異なるモデルを使うことができる
- 線形回帰と決定木回帰では予測の仕組みが異なる
- 学習データの結果だけではなく、テストデータの結果も確認する
- Train と Test の差が大きいときは、過学習を疑う


# 確認問題

次の問いに答えてください。

1. 今回の入力 **X** には、何個の特徴量がありますか。
2. 今回の正解 **y** は何ですか。
3. `MedHouseVal = 3.0` は、何ドルを表しますか。
4. 学習用データとテスト用データは何対何に分けましたか。
5. 線形回帰の **MAE、MSE、RMSE、R²** を書いてください。
6. 決定木回帰の **MAE、RMSE、R²** を書いてください。
7. 線形回帰と決定木回帰では、今回どちらのテスト結果がよかったですか。
8. 深さを制限しない決定木の Train R² と Test R² を比較してください。
9. 8の結果から、過学習について何が言えますか。
10. 分類で使った決定木と、今回の決定木回帰では、最終的に予測するものがどう違いますか。

### 提出するもの
- Linear Regression の MAE / MSE / RMSE / R²
- Decision Tree Regressor の MAE / RMSE / R²
- Actual vs Predicted のグラフ
- 上の確認問題への回答


## 参考

- scikit-learn: California Housing Dataset
- scikit-learn: LinearRegression
- scikit-learn: DecisionTreeRegressor
- scikit-learn: Regression metrics
- データ読み込み用 CSV ミラー: scikit.js GitHub repository

このデータは scikit-learn の California Housing Dataset と同じ
20,640件・8特徴量・1ターゲットのデータです。

この実習では、回帰の基本的な流れとモデル比較を理解することを目的として、
複雑な前処理やハイパーパラメータ探索は行っていません。
